# GIT HUB REPO FETCHING DATA 
# GitHub Python Repos Dataset

A dataset of public GitHub repositories, collected via the GitHub REST API,
cleaned, and enriched with popularity/activity features — ready for model
training tasks like popularity prediction, activity classification, and
repo clustering.

## What's in this repo

| File | Description |
|---|---|
| `github-repo.ipynb` | Full pipeline: fetch (paginated GitHub API calls) → clean → feature engineer → export |
| `github_python_repos.csv` | Final dataset — one row per repository |

## Data source

Collected from the [GitHub Search Repositories API](https://docs.github.com/en/rest/search),
filtered to `language:python`, sorted by star count (descending).

- **Collection date:** snapshot at time of pipeline run (not live/updating)
- **Size:** 300 repositories
- **License:** data reused under GitHub's public API terms — for educational/research use

## Columns

| Column | Type | Description |
|---|---|---|
| `name` | string | Repository name |
| `full_name` | string | `owner/repo` format |
| `owner` | string | Repository owner's GitHub username |
| `description` | string | Repo description (empty string if none) |
| `language` | string | Primary language |
| `stargazers_count` | int | Number of stars |
| `forks_count` | int | Number of forks |
| `watchers_count` | int | Number of watchers |
| `open_issues_count` | int | Open issues at time of collection |
| `created_at` | datetime | Repo creation date |
| `updated_at` | datetime | Last metadata update |
| `pushed_at` | datetime | Last code push |
| `days_since_created` | int | Age of repo in days |
| `days_since_last_push` | int | Days since last commit |
| `stars_per_day` | float | Stars normalized by repo age |
| `fork_to_star_ratio` | float | Forks relative to stars |
| `topic_count` | int | Number of GitHub topic tags |
| `license` | string | License name, or "No License" |
| `size` | int | Repo size (KB) |
| `archived` | bool | Whether the repo is archived |
| `is_active` | int | 1 if pushed within the last 90 days, else 0 |

## Possible use cases

- **Classification:** predict `is_active` from repo metadata
- **Regression:** predict `stargazers_count` or `stars_per_day`
- **Text + tabular:** classify `language` using `description` plus numeric features
- **Clustering:** group repos by popularity/activity signals to find repo "archetypes"
- **Trend analysis:** examine how `stars_per_day` changes with `days_since_created`

## Get the data

Clone the full project:
```bash
git clone https://github.com/AyushPratap05/ML.git
```

Or grab just the CSV: open `github_python_repos.csv` in the repo on GitHub
and use the "Download raw file" option.

## How to use this dataset

**Load it:**
```python
import pandas as pd
df = pd.read_csv("github_python_repos.csv")
```

**Quick baseline — predict `is_active`:**
```python
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

features = ["stargazers_count", "forks_count", "size",
            "days_since_created", "stars_per_day", "fork_to_star_ratio"]
X = df[features]
y = df["is_active"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier()
model.fit(X_train, y_train)
print(model.score(X_test, y_test))
```

Or open `github-repo.ipynb` directly to see the full pipeline and rerun it
with a different search query (e.g. another language) or a larger sample size.

## How to reproduce this pipeline yourself

1. Create a GitHub personal access token: **Settings → Developer settings → Personal access tokens (classic)**
2. Add it to a `.env` file in the project root: `GITHUB_TOKEN=your_token_here`
3. Install dependencies: `pip install requests pandas python-dotenv scikit-learn`
4. Run `github-repo.ipynb` top to bottom

## Limitations

- Fixed snapshot — does not update automatically after collection
- GitHub's Search API caps results at 1000 repos per query
- Limited to repositories tagged `language:python`
- Popularity metrics (stars, forks) can be inflated by factors unrelated to code quality (marketing, timing, etc.)

## License

Released for educational and research use. Original repository data belongs
to GitHub and respective repo owners.

In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

notebook_dir = Path.cwd()  # or hardcode once you find the right folder
load_dotenv(dotenv_path=notebook_dir / ".venv/.env")

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
print(len(GITHUB_TOKEN) if GITHUB_TOKEN else "Token not found")

40


In [3]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd

load_dotenv()  # reads .env from the current working directory and loads it into os.environ

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
BASE_URL = "https://api.github.com"

print(len(GITHUB_TOKEN) if GITHUB_TOKEN else "Token not found")

40


In [4]:
import os
print(os.getcwd())

c:\Users\ayush\PROJECTS\ML


In [5]:
headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

response = requests.get(f"{BASE_URL}/search/repositories",
                         params={"q": "language:python", "sort": "stars", "order": "desc", "per_page": 5},
                         headers=headers)

print(response.status_code)
data = response.json()
data.keys()

200


dict_keys(['total_count', 'incomplete_results', 'items'])

In [6]:
print(data["total_count"])
print(data["incomplete_results"])

first_repo = data["items"][0]
first_repo.keys()

33147384
False


dict_keys(['id', 'node_id', 'name', 'full_name', 'private', 'owner', 'html_url', 'description', 'fork', 'url', 'forks_url', 'keys_url', 'collaborators_url', 'teams_url', 'hooks_url', 'issue_events_url', 'events_url', 'assignees_url', 'branches_url', 'tags_url', 'blobs_url', 'git_tags_url', 'git_refs_url', 'trees_url', 'statuses_url', 'languages_url', 'stargazers_url', 'contributors_url', 'subscribers_url', 'subscription_url', 'commits_url', 'git_commits_url', 'comments_url', 'issue_comment_url', 'contents_url', 'compare_url', 'merges_url', 'archive_url', 'downloads_url', 'issues_url', 'pulls_url', 'milestones_url', 'notifications_url', 'labels_url', 'releases_url', 'deployments_url', 'created_at', 'updated_at', 'pushed_at', 'git_url', 'ssh_url', 'clone_url', 'svn_url', 'homepage', 'size', 'stargazers_count', 'watchers_count', 'language', 'has_issues', 'has_projects', 'has_downloads', 'has_wiki', 'has_pages', 'has_discussions', 'forks_count', 'mirror_url', 'archived', 'disabled', 'open_

In [7]:
fields = {
    "name": first_repo["name"],
    "full_name": first_repo["full_name"],
    "owner": first_repo["owner"]["login"],
    "description": first_repo["description"],
    "language": first_repo["language"],
    "stargazers_count": first_repo["stargazers_count"],
    "forks_count": first_repo["forks_count"],
    "open_issues_count": first_repo["open_issues_count"],
    "watchers_count": first_repo["watchers_count"],
    "created_at": first_repo["created_at"],
    "updated_at": first_repo["updated_at"],
    "pushed_at": first_repo["pushed_at"],
    "size": first_repo["size"],
    "license": first_repo["license"]["name"] if first_repo["license"] else None,
    "topics": first_repo["topics"],
    "archived": first_repo["archived"],
}
fields

{'name': 'public-apis',
 'full_name': 'public-apis/public-apis',
 'owner': 'public-apis',
 'description': 'A collective list of free APIs',
 'language': 'Python',
 'stargazers_count': 475182,
 'forks_count': 52493,
 'open_issues_count': 1887,
 'watchers_count': 475182,
 'created_at': '2016-03-20T23:49:42Z',
 'updated_at': '2026-09-04T13:44:35Z',
 'pushed_at': '2026-09-02T02:23:53Z',
 'size': 9360,
 'license': 'MIT License',
 'topics': ['api',
  'apis',
  'dataset',
  'development',
  'free',
  'list',
  'lists',
  'open-source',
  'public',
  'public-api',
  'public-apis',
  'resources',
  'software'],
 'archived': False}

In [8]:
def extract_repo_fields(repo):
    return {
        "name": repo["name"],
        "full_name": repo["full_name"],
        "owner": repo["owner"]["login"],
        "description": repo["description"],
        "language": repo["language"],
        "stargazers_count": repo["stargazers_count"],
        "forks_count": repo["forks_count"],
        "open_issues_count": repo["open_issues_count"],
        "watchers_count": repo["watchers_count"],
        "created_at": repo["created_at"],
        "updated_at": repo["updated_at"],
        "pushed_at": repo["pushed_at"],
        "size": repo["size"],
        "license": repo["license"]["name"] if repo["license"] else None,
        "topics": repo["topics"],
        "archived": repo["archived"],
    }

rows = [extract_repo_fields(repo) for repo in data["items"]]
df = pd.DataFrame(rows)
df

,name,full_name,owner,description,language,stargazers_count,forks_count,open_issues_count,watchers_count,created_at,updated_at,pushed_at,size,license,topics,archived
0,public-apis,public-apis/public-apis,public-apis,A collective list of free APIs,Python,475182,52493,1887,475182,2016-03-20T23:49:42Z,2026-09-04T13:44:35Z,2026-09-02T02:23:53Z,9360,MIT License,"[api, apis, dataset, development, free, list, ...",False
1,free-programming-books,EbookFoundation/free-programming-books,EbookFoundation,:books: Freely available programming books,Python,395945,66731,82,395945,2013-10-11T06:50:37Z,2026-09-04T13:28:17Z,2026-09-01T09:12:19Z,21792,Creative Commons Attribution 4.0 International,"[books, education, hacktoberfest, list, resource]",False
2,system-design-primer,donnemartin/system-design-primer,donnemartin,Learn how to design large-scale systems. Prep ...,Python,367852,58225,613,367852,2017-02-26T16:15:28Z,2026-09-04T13:44:56Z,2026-03-20T01:52:19Z,11277,Other,"[design, design-patterns, design-system, devel...",False
3,awesome-python,vinta/awesome-python,vinta,"The definitive list that answers ""I want to do...",Python,318174,28632,17,318174,2014-06-27T21:00:06Z,2026-09-04T13:39:40Z,2026-09-01T08:48:12Z,6104,Other,"[awesome, awesome-list, python, python-framewo...",False
4,project-based-learning,practical-tutorials/project-based-learning,practical-tutorials,Curated list of project-based tutorials,Python,282048,36134,278,282048,2017-04-12T05:07:46Z,2026-09-04T13:39:13Z,2026-08-31T07:08:03Z,469,MIT License,"[beginner-project, cpp, golang, javascript, pr...",False


In [9]:
rows[0]

{'name': 'public-apis',
 'full_name': 'public-apis/public-apis',
 'owner': 'public-apis',
 'description': 'A collective list of free APIs',
 'language': 'Python',
 'stargazers_count': 475182,
 'forks_count': 52493,
 'open_issues_count': 1887,
 'watchers_count': 475182,
 'created_at': '2016-03-20T23:49:42Z',
 'updated_at': '2026-09-04T13:44:35Z',
 'pushed_at': '2026-09-02T02:23:53Z',
 'size': 9360,
 'license': 'MIT License',
 'topics': ['api',
  'apis',
  'dataset',
  'development',
  'free',
  'list',
  'lists',
  'open-source',
  'public',
  'public-api',
  'public-apis',
  'resources',
  'software'],
 'archived': False}

In [10]:
df.columns

Index(['name', 'full_name', 'owner', 'description', 'language',
       'stargazers_count', 'forks_count', 'open_issues_count',
       'watchers_count', 'created_at', 'updated_at', 'pushed_at', 'size',
       'license', 'topics', 'archived'],
      dtype='str')

In [11]:
def fetch_repos(query, total_repos=300, per_page=100):
    all_items = []
    pages_needed = (total_repos // per_page) + 1

    for page in range(1, pages_needed + 1):
        response = requests.get(
            f"{BASE_URL}/search/repositories",
            params={"q": query, "sort": "stars", "order": "desc",
                    "per_page": per_page, "page": page},
            headers=headers
        )
        print(f"Page {page}: status {response.status_code}")

        if response.status_code != 200:
            print("Stopping due to error:", response.json())
            break

        page_data = response.json()
        all_items.extend(page_data["items"])

        if len(page_data["items"]) < per_page:
            break  # reached the last available page

    return all_items[:total_repos]

repo_items = fetch_repos("language:python", total_repos=300)
len(repo_items)

Page 1: status 200
Page 2: status 200
Page 3: status 200
Page 4: status 200


300

In [12]:
rows = [extract_repo_fields(repo) for repo in repo_items]
df = pd.DataFrame(rows)
print(df.shape)
df.head()

(300, 16)


,name,full_name,owner,description,language,stargazers_count,forks_count,open_issues_count,watchers_count,created_at,updated_at,pushed_at,size,license,topics,archived
0,public-apis,public-apis/public-apis,public-apis,A collective list of free APIs,Python,475182,52493,1887,475182,2016-03-20T23:49:42Z,2026-09-04T13:44:35Z,2026-09-02T02:23:53Z,9360,MIT License,"[api, apis, dataset, development, free, list, ...",False
1,free-programming-books,EbookFoundation/free-programming-books,EbookFoundation,:books: Freely available programming books,Python,395945,66731,82,395945,2013-10-11T06:50:37Z,2026-09-04T13:28:17Z,2026-09-01T09:12:19Z,21792,Creative Commons Attribution 4.0 International,"[books, education, hacktoberfest, list, resource]",False
2,system-design-primer,donnemartin/system-design-primer,donnemartin,Learn how to design large-scale systems. Prep ...,Python,367852,58225,613,367852,2017-02-26T16:15:28Z,2026-09-04T13:44:56Z,2026-03-20T01:52:19Z,11277,Other,"[design, design-patterns, design-system, devel...",False
3,awesome-python,vinta/awesome-python,vinta,"The definitive list that answers ""I want to do...",Python,318174,28632,17,318174,2014-06-27T21:00:06Z,2026-09-04T13:39:40Z,2026-09-01T08:48:12Z,6104,Other,"[awesome, awesome-list, python, python-framewo...",False
4,project-based-learning,practical-tutorials/project-based-learning,practical-tutorials,Curated list of project-based tutorials,Python,282048,36134,278,282048,2017-04-12T05:07:46Z,2026-09-04T13:39:13Z,2026-08-31T07:08:03Z,469,MIT License,"[beginner-project, cpp, golang, javascript, pr...",False


In [13]:
df.isna().sum()

name                  0
full_name             0
owner                 0
description           4
language              0
stargazers_count      0
forks_count           0
open_issues_count     0
watchers_count        0
created_at            0
updated_at            0
pushed_at             0
size                  0
license              17
topics                0
archived              0
dtype: int64

In [14]:
df.dtypes

name                    str
full_name               str
owner                   str
description             str
language                str
stargazers_count      int64
forks_count           int64
open_issues_count     int64
watchers_count        int64
created_at              str
updated_at              str
pushed_at               str
size                  int64
license                 str
topics               object
archived               bool
dtype: object

In [15]:
df_clean = df.copy()

# Convert date strings to actual datetime objects
date_cols = ["created_at", "updated_at", "pushed_at"]
for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col])

# Fill missing license/description with explicit placeholders instead of NaN
df_clean["license"] = df_clean["license"].fillna("No License")
df_clean["description"] = df_clean["description"].fillna("")

# Drop exact duplicate rows, just in case pagination overlapped
df_clean = df_clean.drop_duplicates(subset="full_name")

print(df_clean.shape)
df_clean.dtypes

(300, 16)


name                                 str
full_name                            str
owner                                str
description                          str
language                             str
stargazers_count                   int64
forks_count                        int64
open_issues_count                  int64
watchers_count                     int64
created_at           datetime64[us, UTC]
updated_at           datetime64[us, UTC]
pushed_at            datetime64[us, UTC]
size                               int64
license                              str
topics                            object
archived                            bool
dtype: object

In [16]:
import datetime

now = pd.Timestamp.now(tz="UTC")

df_feat = df_clean.copy()

df_feat["days_since_created"] = (now - df_feat["created_at"]).dt.days
df_feat["days_since_last_push"] = (now - df_feat["pushed_at"]).dt.days
df_feat["stars_per_day"] = df_feat["stargazers_count"] / df_feat["days_since_created"].replace(0, 1)
df_feat["fork_to_star_ratio"] = df_feat["forks_count"] / df_feat["stargazers_count"].replace(0, 1)
df_feat["topic_count"] = df_feat["topics"].apply(len)
df_feat["is_active"] = (df_feat["days_since_last_push"] <= 90).astype(int)

df_feat[["name", "stars_per_day", "days_since_last_push", "is_active"]].head()

,name,stars_per_day,days_since_last_push,is_active
0,public-apis,124.425766,2,1
1,free-programming-books,84.046911,3,1
2,system-design-primer,105.826237,168,0
3,awesome-python,71.483712,3,1
4,project-based-learning,82.181818,4,1


In [17]:
final_cols = ["name", "full_name", "owner", "description", "language",
              "stargazers_count", "forks_count", "watchers_count", "open_issues_count",
              "created_at", "updated_at", "pushed_at",
              "days_since_created", "days_since_last_push",
              "stars_per_day", "fork_to_star_ratio", "topic_count",
              "license", "size", "archived", "is_active"]

final_df = df_feat[final_cols]
final_df.to_csv("github_python_repos.csv", index=False)
print(f"Saved {len(final_df)} rows")

Saved 300 rows


In [20]:
final_df.shape

(300, 21)

In [21]:
final_df.head()

,name,full_name,owner,description,language,stargazers_count,forks_count,watchers_count,open_issues_count,created_at,...,pushed_at,days_since_created,days_since_last_push,stars_per_day,fork_to_star_ratio,topic_count,license,size,archived,is_active
0,public-apis,public-apis/public-apis,public-apis,A collective list of free APIs,Python,475182,52493,475182,1887,2016-03-20 23:49:42+00:00,...,2026-09-02 02:23:53+00:00,3819,2,124.425766,0.110469,13,MIT License,9360,False,1
1,free-programming-books,EbookFoundation/free-programming-books,EbookFoundation,:books: Freely available programming books,Python,395945,66731,395945,82,2013-10-11 06:50:37+00:00,...,2026-09-01 09:12:19+00:00,4711,3,84.046911,0.168536,5,Creative Commons Attribution 4.0 International,21792,False,1
2,system-design-primer,donnemartin/system-design-primer,donnemartin,Learn how to design large-scale systems. Prep ...,Python,367852,58225,367852,613,2017-02-26 16:15:28+00:00,...,2026-03-20 01:52:19+00:00,3476,168,105.826237,0.158284,13,Other,11277,False,0
3,awesome-python,vinta/awesome-python,vinta,"The definitive list that answers ""I want to do...",Python,318174,28632,318174,17,2014-06-27 21:00:06+00:00,...,2026-09-01 08:48:12+00:00,4451,3,71.483712,0.089988,6,Other,6104,False,1
4,project-based-learning,practical-tutorials/project-based-learning,practical-tutorials,Curated list of project-based tutorials,Python,282048,36134,282048,278,2017-04-12 05:07:46+00:00,...,2026-08-31 07:08:03+00:00,3432,4,82.181818,0.128113,8,MIT License,469,False,1
